# VNAI — ASR + MT (dịch) trên GPU cloud (ezycloudx / pytorch image)

Chạy trên instance GPU dùng template **`pytorch/pytorch:latest`** (CUDA sẵn, WORKDIR `/workspace`). Thay cho notebook Colab cũ.

- **ASR** (whisper-large-v3-turbo) qua WebSocket `/asr`.
- **MT/dịch** (Ollama) qua proxy `/api/*` (cùng server uvicorn).

Khác Colab:
- Không có `/content` → dùng `/workspace`.
- Không cần Cloudflare tunnel → container có IP public, bind thẳng `0.0.0.0:8000`.
  **Mở thêm port `8000` ở tab Storage & Deploy** (mặc định chỉ có 22 + 8888 cho Jupyter).
- Image đã có torch+CUDA → cell 2 chỉ cài `faster-whisper` + FastAPI.
- URL ra là **plain `ws://`/`http://`** (không `wss`/`https` vì không qua Cloudflare).

Chạy tuần tự cell 1→7. Cell 4 chạy lại đến khi log in `ASR ready`. Cell 6 pull model lần đầu ~2-3 phút.

In [ ]:
#@title 1. Kiểm tra GPU
import torch
if not torch.cuda.is_available():
    raise SystemExit('Chưa thấy GPU. Chọn template có CUDA (pytorch/pytorch:latest) rồi chạy lại.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
import subprocess
print('CUDA:', torch.version.cuda)
print('nvidia-smi:'); print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'], capture_output=True, text=True).stdout)

In [ ]:
#@title 2. Cài deps (torch đã có sẵn trong image)
import sys, subprocess
def pip(*a):
    # --break-system-packages: image dùng system Python (/usr/lib/python3.12) bị PEP 668
    # chặn pip install. Flag này cho phép cài vào system env (an toàn trên container throwaway).
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--break-system-packages',*a])
pip('fastapi[standard]','uvicorn[standard]','numpy','faster-whisper','httpx','websockets','requests')
print('Installed.')

In [ ]:
#@title 3. Viết server ASR + proxy Ollama (asr_server.py)
# whisper-large-v3-turbo (cả vi + en). beam_size=1 greedy. vad_filter=True.
# Finalize: temperature fallback + compression/logprob thresholds + fallback partial.
server_code = r'''
import asyncio, json, logging
import numpy as np
import httpx
from contextlib import asynccontextmanager
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, Request
from fastapi.responses import StreamingResponse

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("asr_server")

model = None
OLLAMA_URL = "http://127.0.0.1:11434"

@asynccontextmanager
async def lifespan(app):
    global model
    from faster_whisper import WhisperModel
    logger.info("Loading whisper-large-v3-turbo on GPU (float16)...")
    model = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16")
    logger.info("ASR ready (whisper-large-v3-turbo).")
    app.state.ready = True
    yield
    logger.info("Shutting down ASR server...")

app = FastAPI(lifespan=lifespan)
app.state.ready = False

INITIAL_TRANSCRIBE_BYTES = 6400
PARTIAL_GROW_BYTES = 24000
PARTIAL_FALLBACK_MIN_LOGPROB = -1.0

def _segments_conf(segments):
    segs = list(segments)
    text = " ".join(s.text.strip() for s in segs).strip()
    if not segs:
        return text, 0.0, 1.0, 1.0
    total = 0; wlp = 0.0; nsp = 0.0; cr = 1.0
    for s in segs:
        wc = max(1, len(s.text.split()))
        total += wc
        wlp += (getattr(s, "avg_logprob", 0.0) or 0.0) * wc
        nsp = max(nsp, getattr(s, "no_speech_prob", 0.0) or 0.0)
        cr = max(cr, getattr(s, "compression_ratio", 1.0) or 1.0)
    return text, (wlp / total if total else 0.0), nsp, cr

async def _transcribe(audio_bytes, lang, beam_size=1, fallback=False, prompt=""):
    if not audio_bytes:
        return "", lang, 0.0, 1.0, 1.0
    audio = np.frombuffer(audio_bytes, dtype=np.int16).astype(np.float32) / 32768.0
    loop = asyncio.get_event_loop()
    def run():
        kw = dict(
            language=lang, task="transcribe", beam_size=beam_size,
            condition_on_previous_text=False,
            hallucination_silence_threshold=2.0,
            no_speech_threshold=0.6, vad_filter=True,
        )
        if fallback:
            kw.update(temperature=[0.0, 0.2, 0.4],
                      compression_ratio_threshold=2.4, logprob_threshold=-1.0)
        if prompt:
            kw["initial_prompt"] = prompt
        segments, _ = model.transcribe(audio, **kw)
        return _segments_conf(segments)
    try:
        text, lp, nsp, cr = await loop.run_in_executor(None, run)
        return text, lang, lp, nsp, cr
    except Exception as e:
        logger.exception("transcribe error: %s", e)
        return "", lang, 0.0, 1.0, 1.0

@app.get("/health")
async def health():
    return {"ready": app.state.ready}

@app.websocket("/asr")
async def asr_ws(ws: WebSocket):
    await ws.accept()
    buf = bytearray()
    lang = "vi"
    partial_busy = [False]
    last_partial_len = 0
    last_partial_text = ""
    partial_task = None
    logger.info("ASR client connected.")
    try:
        while True:
            msg = await ws.receive()
            if msg.get("type") == "websocket.disconnect":
                break
            if msg.get("bytes") is not None:
                buf.extend(msg["bytes"])
                grown = len(buf) - last_partial_len
                should_fire = (not partial_busy[0]) and (
                    (last_partial_len == 0 and len(buf) >= INITIAL_TRANSCRIBE_BYTES)
                    or (last_partial_len > 0 and grown >= PARTIAL_GROW_BYTES)
                )
                if should_fire:
                    partial_busy[0] = True
                    last_partial_len = len(buf)
                    audio = bytes(buf)
                    async def _p(audio=audio, lang=lang):
                        nonlocal last_partial_text
                        try:
                            text, _l, lp, nsp, _cr = await _transcribe(audio, lang, beam_size=1)
                            if text:
                                await ws.send_text(json.dumps({"type":"partial","text":text,
                                    "avg_logprob": round(lp,3),
                                    "no_speech_prob": round(nsp,3)}))
                                if lp >= PARTIAL_FALLBACK_MIN_LOGPROB:
                                    last_partial_text = text
                        except Exception as e:
                            logger.exception("partial error: %s", e)
                        finally:
                            partial_busy[0] = False
                    if partial_task and not partial_task.done():
                        partial_task.cancel()
                    partial_task = asyncio.create_task(_p())
            elif msg.get("text"):
                try:
                    data = json.loads(msg["text"])
                except json.JSONDecodeError:
                    continue
                action = data.get("action")
                if action == "start":
                    buf.clear()
                    lang = data.get("lang", "vi")
                    last_partial_len = 0
                    last_partial_text = ""
                    partial_busy[0] = False
                elif action == "finalize":
                    if partial_task and not partial_task.done():
                        partial_task.cancel()
                    text, det, lp, nsp, cr = await _transcribe(
                        bytes(buf), lang, beam_size=1, fallback=True)
                    if not text and last_partial_text:
                        text = last_partial_text
                        lp, nsp, cr = 0.0, 0.0, 1.0
                        logger.info("finalize empty -> fallback partial: %r", text[:80])
                    await ws.send_text(json.dumps({
                        "type": "final", "text": text, "lang": det,
                        "avg_logprob": round(lp, 3),
                        "no_speech_prob": round(nsp, 3),
                        "compression_ratio": round(cr, 3),
                    }))
                    buf.clear()
                    last_partial_len = 0
                    last_partial_text = ""
    except WebSocketDisconnect:
        pass
    except Exception as e:
        logger.exception("asr_ws error: %s", e)
    finally:
        logger.info("ASR client disconnected.")

@app.api_route("/api/{path:path}", methods=["GET", "POST"])
async def proxy_ollama(request: Request, path: str):
    url = f"{OLLAMA_URL}/api/{path}"
    body = await request.body()
    headers = {"Content-Type": request.headers.get("content-type", "application/json")}
    async def stream():
        async with httpx.AsyncClient(timeout=None) as c:
            async with c.stream(request.method, url, content=body, headers=headers) as r:
                async for chunk in r.aiter_raw():
                    yield chunk
    return StreamingResponse(stream(), media_type="application/x-ndjson",
        headers={"Cache-Control":"no-cache","X-Accel-Buffering":"no"})
'''
import os
os.makedirs('/workspace', exist_ok=True)
open('/workspace/asr_server.py','w').write(server_code)
print('Wrote /workspace/asr_server.py')

In [ ]:
#@title 4. (Re)khởi động uvicorn (nền) — kill uvicorn cũ trước
import subprocess, os, time, sys
PORT = 8000  #@param {type:"integer"}
subprocess.run(['pkill','-f','asr_server'], check=False)
time.sleep(1)
env = os.environ.copy()
env['PYTHONPATH'] = '/workspace'
logf = open('/workspace/asr_server.log','w')
# Dùng sys.executable -m uvicorn: luôn dùng đúng Python của kernel (nơi cell 2 đã cài uvicorn),
# không phụ thuộc PATH -> tránh FileNotFoundError khi 'uvicorn' không trên PATH.
server_proc = subprocess.Popen(
    [sys.executable,'-m','uvicorn','asr_server:app','--host','0.0.0.0','--port',str(PORT)],
    cwd='/workspace', env=env, stdout=logf, stderr=subprocess.STDOUT)
print('uvicorn PID:', server_proc.pid, 'on 0.0.0.0:%d' % PORT)
print('(download whisper-large-v3-turbo lần đầu ~1-2 phút)')

In [ ]:
#@title 5. Xem log server (chạy lại đến khi thấy "ASR ready")
import time; time.sleep(2)
print(open('/workspace/asr_server.log').read()[-2500:])

In [ ]:
#@title 6. Cài + chạy Ollama (GPU), pull model dịch
import subprocess, time, urllib.request, shutil, os, json, requests

# === Đổi model dịch tại đây ===
MT_MODEL_NAME = 'gemma3:4b'  # vd: 'qwen2.5:7b','qwen2.5:3b','qwen2.5:14b'

subprocess.run(['apt-get','update','-qq'], check=False)
subprocess.run(['apt-get','install','-y','zstd'], check=False)
print('Running ollama install.sh ...')
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=False)
for _ in range(30):
    if shutil.which('ollama') or os.path.exists('/usr/local/bin/ollama'):
        break
    time.sleep(1)
ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
if not os.path.exists(ollama_bin):
    raise SystemExit('Cài ollama thất bại - không thấy binary.')
print('ollama binary:', ollama_bin)
ollama_proc = subprocess.Popen(['ollama','serve'],
    stdout=open('/workspace/ollama.log','w'), stderr=subprocess.STDOUT)
time.sleep(6)
try:
    r = urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=10)
    print('Ollama server UP:', r.status)
except Exception as e:
    print('Ollama chưa lên. Log:'); print(open('/workspace/ollama.log').read()[-2000:])
    raise SystemExit(str(e))

print(f'Pulling {MT_MODEL_NAME} ...')
with requests.post('http://127.0.0.1:11434/api/pull',
                   json={'name': MT_MODEL_NAME, 'stream': True},
                   stream=True, timeout=None) as resp:
    resp.raise_for_status()
    last = ''
    for line in resp.iter_lines():
        if not line:
            continue
        d = json.loads(line)
        st = d.get('status', '')
        total = d.get('total'); completed = d.get('completed')
        # Guard: một số dòng (verifying/sha256) có total nhưng completed=None
        if total and completed is not None:
            pct = completed / total * 100
            last = f'{st}: {pct:5.1f}% ({completed/1e9:.2f}/{total/1e9:.2f} GB)'
            print('\r' + last, end='', flush=True)
        else:
            if st != 'pulling manifest' and st != last:
                print('\n' + st, flush=True)
                last = st
print('\nPull done:', MT_MODEL_NAME)

print('Warm-up load vào VRAM ...')
try:
    w = requests.post('http://127.0.0.1:11434/api/chat',
        json={'model': MT_MODEL_NAME,
              'messages': [{'role':'user','content':'ok'}],
              'stream': False, 'options': {'num_predict': 1}},
        timeout=120)
    print('warm-up status:', w.status_code)
except Exception as e:
    print('warm-up lỗi (không fatal, model vẫn pull xong):', e)

print('Ollama ready at 127.0.0.1:11434')
res = subprocess.run(['ollama','ps'], capture_output=True, text=True)
print(res.stdout)
# Kiểm tra ollama có thấy GPU không (WARN ở install là chỉ script auto-detect, không phải lúc chạy)
print('--- ollama.log (GPU?) ---')
print(open('/workspace/ollama.log').read()[-1500:])

In [ ]:
#@title 7. In link public (RunPod proxy / IP public)
import os, urllib.request

PORT = 8000  #@param {type:"integer"}

# Kiểm tra health cục bộ trước
try:
    h = urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health', timeout=5)
    print('Local /health:', h.read().decode())
except Exception as e:
    print('Local /health chưa lên — chạy lại cell 4-5 đến khi "ASR ready":', e)

print()
base = None
# RunPod: tự proxy mọi port theo dạng https://<pod_id>-<port>.proxy.runpod.net (TLS sẵn, cố định).
pod_id = os.getenv('RUNPOD_POD_ID')
if pod_id:
    base = f'https://{pod_id}-{PORT}.proxy.runpod.net'
    print('RunPod pod:', pod_id)

if not base:
    # Fallback: IP public (chỉ dùng được nếu platform đã mở port này ra ngoài).
    try:
        ip = urllib.request.urlopen('https://api.ipify.org', timeout=10).read().decode().strip()
        base = f'http://{ip}:{PORT}'
        print('Public IP:', ip, '(cần port', PORT, 'được mở ra ngoài — nếu timeout thì chạy cell 7b cloudflared)')
    except Exception as e:
        print('Không lấy được IP public:', e)

if base:
    print('HTTP health:', base + '/health')
    print()
    print('>>> ASR_REMOTE_URL cho .env:'); print(base.replace('http://','ws://').replace('https://','wss://') + '/asr')
    print()
    print('>>> MT_BASE_URL cho .env:'); print(base)
    print('>>> MT_MODEL cho .env:', MT_MODEL_NAME)

In [ ]:
#@title 7b. (Nếu port 8000 bị chặn) Cloudflare Tunnel — không cần mở port
# Chạy cell này KHI port 8000 không ra được internet (test http://<IP>:8000/health từ máy ngoài timeout).
# Cloudflared phô localhost:8000 ra 1 URL trycloudflare (wss/https, ổn định cho WS streaming).
import subprocess, time, re, os

CF_BIN = '/usr/local/bin/cloudflared'
if not os.path.exists(CF_BIN):
    print('Downloading cloudflared ...')
    for attempt in range(4):
        r = subprocess.run(['curl','-L','--retry','3','--retry-delay','2','-f',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            '-o', CF_BIN])
        if r.returncode == 0:
            break
        print(f'download attempt {attempt+1} failed (exit {r.returncode}), retry...')
        time.sleep(3)
    if r.returncode != 0:
        subprocess.run(['curl','-L','-f',
            'https://github.com/cloudflare/cloudflared/releases/download/2025.7.1/cloudflared-linux-amd64',
            '-o', CF_BIN], check=True)
    subprocess.run(['chmod','+x', CF_BIN], check=True)
else:
    print('cloudflared đã có, skip download.')

try:
    cf_proc.send_signal(15); time.sleep(2)
except Exception:
    pass
cf_log = open('/workspace/cloudflared.log','w')
cf_proc = subprocess.Popen([CF_BIN,'tunnel','--url','http://localhost:8000'],
    stdout=cf_log, stderr=subprocess.STDOUT)

base_url = None
for _ in range(60):
    time.sleep(1)
    try:
        txt = open('/workspace/cloudflared.log').read()
    except Exception:
        txt = ''
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
    if m:
        base_url = m.group(0); break
if not base_url:
    print('Không lấy được URL. Log:'); print(open('/workspace/cloudflared.log').read()[-2000:])
    raise SystemExit('Tunnel chưa sẵn sàng.')

ASR_URL = base_url.replace('https://','wss://') + '/asr'
MT_BASE_URL = base_url
print('HTTP health:', base_url + '/health')
print()
print('>>> ASR_REMOTE_URL cho .env:'); print(ASR_URL)
print()
print('>>> MT_BASE_URL cho .env:'); print(MT_BASE_URL)
print('>>> MT_MODEL cho .env:', MT_MODEL_NAME)
print()
print('(URL Cloudflare đổi mỗi lần chạy cell này -> cập nhật .env + restart backend)')

In [ ]:
#@title 8. Test nhanh ASR WS (qua RunPod proxy / tunnel / IP)
import numpy as np, asyncio, json, websockets, re, os, urllib.request

PORT = 8000
url = None
# 1) RunPod proxy (TLS, cố định)
pod_id = os.getenv('RUNPOD_POD_ID')
if pod_id:
    url = f'wss://{pod_id}-{PORT}.proxy.runpod.net/asr'
# 2) cloudflared (cell 7b)
if not url:
    try:
        txt = open('/workspace/cloudflared.log').read()
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
        if m: url = m.group(0).replace('https://','wss://') + '/asr'
    except Exception:
        pass
# 3) IP public
if not url:
    ip = urllib.request.urlopen('https://api.ipify.org', timeout=10).read().decode().strip()
    url = f'ws://{ip}:{PORT}/asr'

print('Testing', url)
async def test():
    sr=16000; t=np.arange(int(sr*2))/sr
    tone=(0.3*np.sin(2*np.pi*440*t)*32767).astype(np.int16).tobytes()
    async with websockets.connect(url, max_size=None) as ws:
        await ws.send(json.dumps({'action':'start','lang':'vi'}))
        for i in range(0, len(tone), 1600):
            await ws.send(tone[i:i+1600]); await asyncio.sleep(0.05)
        await ws.send(json.dumps({'action':'finalize'}))
        while True:
            raw=await asyncio.wait_for(ws.recv(), timeout=30)
            print('<=', raw[:200])
            if json.loads(raw).get('type')=='final': break
await test()
print('OK — WS ASR hoạt động.')

## Dùng với backend

Sau khi cell 7 in ra 3 giá trị, điền vào `D:/VNAI/.env` (hoặc env của backend):

```env
ASR_REMOTE_URL=ws://<IP>:8000/asr
MT_BASE_URL=http://<IP>:8000
MT_MODEL=gemma3:4b   # đúng với MT_MODEL_NAME ở cell 6
```

Khác Colab: URL **ổn định** (IP cố định của instance, không đổi mỗi lần chạy như Cloudflare). Chỉ cần cập nhật khi instance restart/get IP mới.

Chạy backend:
```powershell
python -m uvicorn backend.main:app --host 0.0.0.0 --port 8000
```

- `ASR_REMOTE_URL` set → ASR gọi GPU cloud. Để trống → whisper-medium local CPU.
- `MT_BASE_URL` set → dịch gọi Ollama trên GPU cloud. Để trống → Ollama local.
- VAD + TTS vẫn chạy local (backend).

In [ ]:
#@title Dừng tất cả (uvicorn + ollama)
import signal
for name, var in [('uvicorn','server_proc'),('ollama','ollama_proc')]:
    try:
        var.send_signal(signal.SIGTERM); print(name, 'stopped')
    except Exception as e:
        print(name, '?', e)